# 2.2 ML model quality

Classification, regression and ranking quality reports with Evidently 0.7.

In [ ]:
import numpy as np

# Report runs metrics; Dataset + DataDefinition tell Evidently what each column is
from evidently import BinaryClassification, DataDefinition, Dataset, Regression, Report

# Presets = ready-made bundles of metrics and plots
from evidently.presets import ClassificationPreset, RegressionPreset
from sklearn import datasets, ensemble

## Classification

Breast cancer dataset, random forest, predicted probabilities.

In [ ]:
# Breast cancer dataset: 30 numeric features, binary target (0 = malignant, 1 = benign)
data = datasets.load_breast_cancer(as_frame=True)
df = data.frame
features = data.feature_names.tolist()

# Two random samples: reference = data the model was trained on, current = "production" data
reference = df.sample(n=300, random_state=1)
current = df.sample(n=200, random_state=2)

# Train a small random forest on the reference data
model = ensemble.RandomForestClassifier(n_estimators=10, random_state=1)
model.fit(reference[features], reference.target)

# Store the predicted probability of class 1 as the "prediction" column for both sets
reference["prediction"] = model.predict_proba(reference[features])[:, 1]
current["prediction"] = model.predict_proba(current[features])[:, 1]

# Look at 5 random rows: true label vs predicted probability
print("Reference")
display(reference[["target", "prediction"]].sample(5, random_state=0))

In [ ]:
# Tell Evidently which column is the true label and which holds the predicted probability
cls_definition = DataDefinition(
    classification=[BinaryClassification(target="target", prediction_probas="prediction", pos_label=1)], # pos_label=1 tells Evidently which class is the "positive" one. (required for precision, recall etc)
    numerical_columns=features,
)

# Probability >= 0.7 counts as class 1
cls_report = Report([ClassificationPreset(probas_threshold=0.7)])

# Compare current against reference; the result renders as an interactive report
cls_result = cls_report.run(
    current_data=Dataset.from_pandas(current, data_definition=cls_definition),
    reference_data=Dataset.from_pandas(reference, data_definition=cls_definition),
)
cls_result

## Regression

California housing with a fake prediction (target + noise).

In [ ]:
# California housing: 8 numeric features, target = median house value
housing = datasets.fetch_california_housing(as_frame=True).frame
housing = housing.rename(columns={"MedHouseVal": "target"})

# No real model here: fake predictions = true value + random noise
housing["prediction"] = housing["target"] + np.random.normal(0, 0.5, len(housing))

# Two random samples to act as reference and current data
housing_reference = housing.sample(5000, random_state=1)
housing_current = housing.sample(5000, random_state=2)

In [ ]:
# Tell Evidently which columns are the true value and the prediction
reg_definition = DataDefinition(regression=[Regression(target="target", prediction="prediction")])

# RegressionPreset: MAE, RMSE, MAPE, error plots, predicted vs actual
reg_report = Report([RegressionPreset()])
reg_result = reg_report.run(
    current_data=Dataset.from_pandas(housing_current, data_definition=reg_definition),
    reference_data=Dataset.from_pandas(housing_reference, data_definition=reg_definition),
)
reg_result

## Ranking

Synthetic recommendations: 10 scored items per user. `target` = 1 if the item is relevant.
The current model's scores are less tied to true relevance than the reference, so metrics should drop.

In [ ]:
import pandas as pd
from evidently import Recsys
from evidently.metrics import PrecisionTopK, RecallTopK, NDCG, MAP, MRR, HitRate

rng = np.random.default_rng(0)

# Build fake recommendations: each user gets 10 random items out of 100
def make_recs(n_users, quality):
    rows = []
    for user in range(n_users):
        items = rng.choice(100, size=10, replace=False)
        # Hidden true relevance of each item for this user
        relevance = rng.random(10)
        # Model score: mix of true relevance and noise. Higher quality = less noise
        score = quality * relevance + (1 - quality) * rng.random(10)
        for item, rel, s in zip(items, relevance, score):
            # target = 1 if the item is actually relevant
            rows.append({"user_id": user, "item_id": item, "prediction": s, "target": int(rel > 0.7)})
    return pd.DataFrame(rows)

# Reference = good model, current = worse model, so ranking metrics should drop
recs_reference = make_recs(200, quality=0.8)
recs_current = make_recs(200, quality=0.4)
recs_current.head(10)

In [ ]:
# Tell Evidently which columns identify the user, the item, the true relevance and the score
rank_definition = DataDefinition(
    numerical_columns=["prediction", "target"],
    categorical_columns=["user_id", "item_id"],
    ranking=[Recsys(user_id="user_id", item_id="item_id", target="target", prediction="prediction")],
)

# Evaluate the top 5 items per user
k = 5
rank_report = Report([PrecisionTopK(k=k), RecallTopK(k=k), NDCG(k=k), MAP(k=k), MRR(k=k), HitRate(k=k)])
rank_result = rank_report.run(
    current_data=Dataset.from_pandas(recs_current, data_definition=rank_definition),
    reference_data=Dataset.from_pandas(recs_reference, data_definition=rank_definition),
)
rank_result

## Output

In [ ]:
# Save each report as a standalone HTML file to open in a browser or share
cls_result.save_html("classification_report.html")
reg_result.save_html("regression_report.html")
rank_result.save_html("ranking_report.html")

In [ ]:
# All metric values as a JSON string (first 500 characters shown)
cls_result.json()[:500]

In [ ]:
# Same results as a Python dict, e.g. to log metrics to a database
d = cls_result.dict()
d.keys()